In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse
from pathlib import Path
import re
import random
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### EEGNet-8,2

In [3]:
from pathlib import Path
import re, random, math, os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, roc_curve,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.constraints import max_norm   # used in DeepConvNet

# =========================
# Config
# =========================
SPLIT_DIR = r"/content/drive/MyDrive/CD/patient_data_clean_1800s_nozero_181920212223_in3days_v2"
POS_PATIENTS = {1, 2, 16, 19, 21, 22, 25, 37, 39, 43, 44, 47, 50, 56, 58, 62, 65, 66, 73, 78}

BATCH_SIZE       = 3
EPOCHS           = 100
LR               = 1e-4
SEED             = 1
K_FOLDS          = 7
BEST_MODEL_TPL   = "best_fold_{:02d}.h5"

# not used for dedup height; kept for reference
TARGET_H = 300

# =========================
# Noise config (train-time augmentation only)
# =========================
TRAIN_ADD_GAUSS_NOISE = True     # training generator only
TRAIN_NOISE_FRAC      = 0.10
TRAIN_NOISE_PROB      = 1.0
EPS_STD               = 1e-8

# =========================
# Repro
# =========================
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
for g in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

# =========================
# Helpers for ID/labels
# =========================
PATIENT_NUM_RX = re.compile(r'^ID(\d+)')

def patient_num_from_path(pathlike):
    stem = Path(pathlike).stem
    m = PATIENT_NUM_RX.match(stem)
    return int(m.group(1)) if m else None

def label_for_file(p: Path) -> int:
    pnum = patient_num_from_path(p)
    return 1 if (pnum is not None and pnum in POS_PATIENTS) else 0

# =========================
# List files (all) & labels by patient ID
# =========================
split_dir = Path(SPLIT_DIR)
all_csvs = sorted(split_dir.glob("*.csv"))
if not all_csvs:
    raise FileNotFoundError(f"No CSV found in {SPLIT_DIR}")

id_to_files = {}
for f in all_csvs:
    pid = patient_num_from_path(f)
    if pid is None or pid == 12:    # exclude ID 12
        continue
    id_to_files.setdefault(pid, []).append(f)

all_ids = sorted(id_to_files.keys())
id_labels = np.array([1 if pid in POS_PATIENTS else 0 for pid in all_ids], dtype=int)
print("Total IDs:", len(all_ids), "| Pos IDs:", id_labels.sum(), "| Neg IDs:", (1 - id_labels).sum())

# =========================
# Column handling
# =========================
def _drop_time_cols(df: pd.DataFrame) -> pd.DataFrame:
    time_like = [c for c in df.columns if isinstance(c, str) and c.strip().lower() == "time"]
    return df.drop(columns=time_like, errors="ignore")

def _to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    df = _drop_time_cols(df)
    return df.apply(pd.to_numeric, errors="coerce").fillna(0.0)

def _pad_crop_2d(x: np.ndarray, H: int, W: int) -> np.ndarray:
    h, w = x.shape
    if h < H:
        pad = np.zeros((H, w), dtype=x.dtype); pad[:h, :] = x; x = pad; h = H
    elif h > H:
        x = x[:H, :]; h = H
    if w < W:
        pad = np.zeros((h, W), dtype=x.dtype); pad[:, :w] = x; x = pad
    elif w > W:
        x = x[:, :W]
    return x

def _infer_target_width(example_csv: Path) -> int:
    df = pd.read_csv(example_csv)
    df2 = _to_numeric_df(df)
    return df2.shape[1]

# Fix width/height ONCE using the whole dataset (so all folds match)
TARGET_W = _infer_target_width(all_csvs[0])
print(f"TARGET_W={TARGET_W} (non-Time columns)")

def _dedupe_consecutive_rows(mat: np.ndarray) -> np.ndarray:
    if mat.size == 0:
        return mat
    if mat.ndim == 1:
        mat = mat[:, None]
    if mat.shape[0] == 1:
        return mat
    diffs = np.any(mat[1:] != mat[:-1], axis=1)
    keep = np.concatenate(([True], diffs))
    return mat[keep]

def _dedup_len_from_csv(p: Path) -> int:
    df = pd.read_csv(p)
    mat = _to_numeric_df(df).to_numpy(dtype=np.float32)
    mat_d = _dedupe_consecutive_rows(mat)
    return int(mat_d.shape[0])

TARGET_H_DEDUP = max(_dedup_len_from_csv(f) for f in all_csvs)
if TARGET_H_DEDUP <= 0:
    raise RuntimeError("After de-dup, zero-length found in data.")
print(f"Fixed input height after de-dup (all IDs): H={TARGET_H_DEDUP}")

def load_csv_as_image_dedup(csv_path: Path) -> np.ndarray:
    df  = pd.read_csv(csv_path)
    df2 = _to_numeric_df(df)
    mat = df2.to_numpy(dtype=np.float32)      # shape ~ (Samples, Chans)
    if mat.ndim != 2:
        mat = mat.reshape(mat.shape[0], -1) if mat.ndim > 2 else mat
    mat = _dedupe_consecutive_rows(mat)
    if mat.shape[0] == 0:
        mat = np.zeros((1, TARGET_W), dtype=np.float32)
    mat = _pad_crop_2d(mat, TARGET_H_DEDUP, TARGET_W)
    # keep shape as (Samples, Chans); model will permute internally
    img = np.expand_dims(mat, axis=-1).astype(np.float32)  # (Samples, Chans, 1)
    if img.shape != (TARGET_H_DEDUP, TARGET_W, 1):
        raise ValueError(f"loader produced {img.shape}, expected {(TARGET_H_DEDUP, TARGET_W, 1)} for {csv_path}")
    return img

# cache
CACHE_DEDUP = {}
def load_csv_as_image_cached(csv_path: Path) -> np.ndarray:
    key = ("dedup", str(csv_path))
    if key in CACHE_DEDUP:
        return CACHE_DEDUP[key]
    img = load_csv_as_image_dedup(csv_path)
    CACHE_DEDUP[key] = img
    return img

print("Normalization: NONE. De-dup: remove consecutive duplicate rows (row-wise).")

# =========================
# Keras Sequence
# =========================
class ImageSequence(keras.utils.Sequence):
    def __init__(self, files, batch_size=BATCH_SIZE, shuffle=True,
                 add_noise=False, noise_frac=TRAIN_NOISE_FRAC, noise_prob=TRAIN_NOISE_PROB):
        super().__init__()
        self.files = list(files)
        self.batch_size = int(batch_size)
        self.shuffle = shuffle
        self.add_noise = add_noise
        self.noise_frac = float(noise_frac)
        self.noise_prob = float(noise_prob)
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.files) / self.batch_size)

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.files))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, idx):
        idxs = self.indexes[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_files = [self.files[i] for i in idxs]
        B = len(batch_files)
        X = np.empty((B, TARGET_H_DEDUP, TARGET_W, 1), dtype=np.float32)  # (Samples, Chans, 1)
        y = np.empty((B,), dtype=np.int32)

        for i, f in enumerate(batch_files):
            xi = load_csv_as_image_cached(f)
            if xi.ndim == 2:
                xi = xi[..., None]
            if xi.shape != (TARGET_H_DEDUP, TARGET_W, 1):
                raise ValueError(f"Sample shape {xi.shape} for {f}; expected {(TARGET_H_DEDUP, TARGET_W, 1)}")
            X[i] = xi
            y[i] = label_for_file(f)

        if self.add_noise and self.noise_frac > 0.0 and self.noise_prob > 0.0:
            samp_std = X.reshape(B, -1).std(axis=1).astype(np.float32)
            samp_std = np.maximum(samp_std, EPS_STD).reshape(B, 1, 1, 1)
            noise = np.random.normal(0.0, 1.0, size=X.shape).astype(np.float32)
            noise *= (self.noise_frac * samp_std)
            if self.noise_prob < 1.0:
                mask = (np.random.rand(B, 1, 1, 1) < self.noise_prob).astype(np.float32)
                noise *= mask
            X = X + noise

        if X.shape[1:] != (TARGET_H_DEDUP, TARGET_W, 1):
            raise ValueError(f"Batch X has shape {X.shape}; expected (B, {TARGET_H_DEDUP}, {TARGET_W}, 1)")
        return X, y

# =========================
# DeepConvNet model (binary head)
# =========================
def build_model(samples=TARGET_H_DEDUP, chans=TARGET_W, lr=LR,
                F1=25, F2=50, F3=100, F4=200,
                dropout=0.5):
    """
    DeepConvNet (Schirrmeister et al. 2017) adapted for binary classification.
    Input: (Samples, Chans, 1), internally permuted to (Chans, Samples, 1).
    """
    inputs = keras.Input(shape=(samples, chans, 1))  # (T, C, 1)

    # Permute to (Chans, Samples, 1)
    x = layers.Permute((2, 1, 3))(inputs)  # (C, T, 1)

    # Block 1: temporal conv + spatial conv
    x = layers.Conv2D(
        F1, (1, 5),
        padding="same",
        use_bias=False
    )(x)
    x = layers.Conv2D(
        F1, (chans, 1),  # across all channels
        padding="valid",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("elu")(x)
    x = layers.MaxPooling2D(pool_size=(1, 2))(x)
    x = layers.Dropout(dropout)(x)

    # Block 2
    x = layers.Conv2D(
        F2, (1, 5),
        padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("elu")(x)
    x = layers.MaxPooling2D(pool_size=(1, 2))(x)
    x = layers.Dropout(dropout)(x)

    # Block 3
    x = layers.Conv2D(
        F3, (1, 5),
        padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("elu")(x)
    x = layers.MaxPooling2D(pool_size=(1, 2))(x)
    x = layers.Dropout(dropout)(x)

    # Block 4
    x = layers.Conv2D(
        F4, (1, 5),
        padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("elu")(x)
    x = layers.MaxPooling2D(pool_size=(1, 2))(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, kernel_constraint=max_norm(0.5))(x)
    x = layers.Activation("elu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[keras.metrics.BinaryAccuracy(name="acc"),
                 keras.metrics.AUC(name="auc")],
    )
    return model

# =========================
# Metric helpers (for TEST)
# =========================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0

def youden_threshold_from_probs(y_true, y_prob):
    """Return best threshold and J on this set."""
    if len(np.unique(y_true)) < 2:
        return 0.5, 0.0
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    j = tpr - fpr
    i = int(np.argmax(j))
    return float(thr[i]), float(j[i])

def mean_std(x):
    x = np.asarray(x, dtype=float)
    return np.nanmean(x), np.nanstd(x)

# =========================
# 7-fold CV by patient ID
# =========================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

# Lists for TEST metrics (@0.5 and @Youden-J)
AUC_list,  ACC_list,  PREC_list,  REC_list,  F1_list,  SPEC_list  = [], [], [], [], [], []
AUC_J_list, ACC_J_list, PREC_J_list, REC_J_list, F1_J_list, SPEC_J_list = [], [], [], [], [], []

rows = []

print(f"\n=== {K_FOLDS}-Fold CV (by patient ID) ===")
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(all_ids, id_labels), start=1):
    ids_train_full = [all_ids[i] for i in train_idx]
    ids_test       = [all_ids[i] for i in test_idx]

    # small validation split from training IDs (for checkpointing)
    train_labels_full = np.array([1 if pid in POS_PATIENTS else 0 for pid in ids_train_full], dtype=int)
    try:
        ids_tr, ids_val = train_test_split(
            ids_train_full, test_size=0.10, random_state=SEED, stratify=train_labels_full
        )
    except ValueError:
        ids_tr, ids_val = train_test_split(ids_train_full, test_size=0.10, random_state=SEED, shuffle=True)
        print(f"(Fold {fold_idx}) Warning: stratified VAL split failed; using unstratified split.")

    # Files for each split
    train_files = [f for pid in ids_tr  for f in id_to_files[pid]]
    val_files   = [f for pid in ids_val for f in id_to_files[pid]]
    test_files  = [f for pid in ids_test for f in id_to_files[pid]]

    def split_summary(name, ids, files):
        ys = np.array([label_for_file(f) for f in files], dtype=int)
        print(f"{name:>6} | ids: {len(ids):4d} | files: {len(files):5d} | pos files: {(ys==1).sum():4d} | neg files: {(ys==0).sum():4d}")

    print(f"\n--- Fold {fold_idx}/{K_FOLDS} ---")
    split_summary("train", ids_tr,  train_files)
    split_summary("val",   ids_val, val_files)
    split_summary("test",  ids_test, test_files)

    # Generators
    train_gen = ImageSequence(train_files, batch_size=BATCH_SIZE, shuffle=True,
                              add_noise=TRAIN_ADD_GAUSS_NOISE,
                              noise_frac=TRAIN_NOISE_FRAC,
                              noise_prob=TRAIN_NOISE_PROB)
    val_gen   = ImageSequence(val_files,   batch_size=BATCH_SIZE, shuffle=False, add_noise=False)
    test_gen  = ImageSequence(test_files,  batch_size=BATCH_SIZE, shuffle=False, add_noise=False)

    # Train & pick best by val_loss
    model = build_model()
    best_path = BEST_MODEL_TPL.format(fold_idx)
    ckpt = keras.callbacks.ModelCheckpoint(best_path, monitor="val_loss", mode="min",
                                           save_best_only=True, verbose=1)
    _ = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=[ckpt], verbose=1)

    # Evaluate on TEST
    best_model = keras.models.load_model(best_path)
    test_probs = best_model.predict(test_gen, verbose=0).ravel().astype(float)
    test_y     = np.array([label_for_file(f) for f in test_gen.files], dtype=int)

    # AUC (threshold-free)
    try:
        test_auc = roc_auc_score(test_y, test_probs)
    except ValueError:
        test_auc = float('nan')

    # --- Threshold 0.5 ---
    y_pred_05 = (test_probs >= 0.5).astype(int)
    acc_05  = accuracy_score(test_y, y_pred_05)
    prec_05 = precision_score(test_y, y_pred_05, zero_division=0)
    rec_05  = recall_score(test_y, y_pred_05, zero_division=0)
    f1_05   = f1_score(test_y, y_pred_05, zero_division=0)
    spec_05 = specificity_score(test_y, y_pred_05)

    # --- Youden-J threshold on TEST ---
    t_star, J_val = youden_threshold_from_probs(test_y, test_probs)
    y_pred_J = (test_probs >= t_star).astype(int)
    acc_J  = accuracy_score(test_y, y_pred_J)
    prec_J = precision_score(test_y, y_pred_J, zero_division=0)
    rec_J  = recall_score(test_y, y_pred_J, zero_division=0)
    f1_J   = f1_score(test_y, y_pred_J, zero_division=0)
    spec_J = specificity_score(test_y, y_pred_J)

    # collect lists
    AUC_list.append(test_auc); ACC_list.append(acc_05);   PREC_list.append(prec_05)
    REC_list.append(rec_05);   F1_list.append(f1_05);     SPEC_list.append(spec_05)

    AUC_J_list.append(test_auc); ACC_J_list.append(acc_J);   PREC_J_list.append(prec_J)
    REC_J_list.append(rec_J);    F1_J_list.append(f1_J);     SPEC_J_list.append(spec_J)

    print(f"Fold {fold_idx} | TEST@0.5  ACC={acc_05:.4f} | PREC={prec_05:.4f} | "
          f"REC={rec_05:.4f} | F1={f1_05:.4f} | AUC={test_auc:.4f} | SPEC={spec_05:.4f} | n={len(test_y)}")
    print(f"Fold {fold_idx} | TEST@J    ACC={acc_J:.4f} | PREC={prec_J:.4f} | "
          f"REC={rec_J:.4f} | F1={f1_J:.4f} | AUC={test_auc:.4f} | SPEC={spec_J:.4f} | thr={t_star:.4f}, J={J_val:.4f}")
    print("Confusion matrix (@0.5):\n", confusion_matrix(test_y, y_pred_05))
    print("Classification report (@0.5):\n", classification_report(test_y, y_pred_05, digits=3))

    rows.append({
        "fold": fold_idx,
        "train_files": len(train_files),
        "val_files": len(val_files),
        "test_files": len(test_files),

        "ACC_0.5": acc_05,
        "PREC_0.5": prec_05,
        "REC_0.5": rec_05,
        "F1_0.5": f1_05,
        "AUC": test_auc,
        "SPEC_0.5": spec_05,

        "thr_J": t_star,
        "J": J_val,
        "ACC_J": acc_J,
        "PREC_J": prec_J,
        "REC_J": rec_J,
        "F1_J": f1_J,
        "SPEC_J": spec_J,
    })

# =========================
# Summary across folds
# =========================
print("\n=== TEST metrics per fold — threshold 0.5 ===")
print("AUC :", [None if np.isnan(x) else round(x,4) for x in AUC_list])
print("ACC :", [round(x,4) for x in ACC_list])
print("PREC:", [round(x,4) for x in PREC_list])
print("REC :", [round(x,4) for x in REC_list])
print("SPEC:", [round(x,4) for x in SPEC_list])
print("F1  :", [round(x,4) for x in F1_list])

print("\n=== TEST metrics per fold — Youden-J (on TEST) ===")
print("AUC@J :", [None if np.isnan(x) else round(x,4) for x in AUC_J_list])
print("ACC@J :", [round(x,4) for x in ACC_J_list])
print("PREC@J:", [round(x,4) for x in PREC_J_list])
print("REC@J :", [round(x,4) for x in REC_J_list])
print("SPEC@J:", [round(x,4) for x in SPEC_J_list])
print("F1@J  :", [round(x,4) for x in F1_J_list])

mAUC,  sAUC  = mean_std(AUC_list)
mACC,  sACC  = mean_std(ACC_list)
mPREC, sPREC = mean_std(PREC_list)
mREC,  sREC  = mean_std(REC_list)
mF1,   sF1   = mean_std(F1_list)
mSPEC, sSPEC = mean_std(SPEC_list)

print("\n=== Mean ± SD across folds (TEST) — 0.5 ===")
print(f"{'AUC':>12}: {mAUC:.4f} ± {sAUC:.4f}")
print(f"{'ACC':>12}: {mACC:.4f} ± {sACC:.4f}")
print(f"{'Precision':>12}: {mPREC:.4f} ± {sPREC:.4f}")
print(f"{'Recall':>12}: {mREC:.4f} ± {sREC:.4f}")
print(f"{'Specificity':>12}: {mSPEC:.4f} ± {sSPEC:.4f}")
print(f"{'F1':>12}: {mF1:.4f} ± {sF1:.4f}")

mAUC_J,  sAUC_J  = mean_std(AUC_J_list)
mACC_J,  sACC_J  = mean_std(ACC_J_list)
mPREC_J, sPREC_J = mean_std(PREC_J_list)
mREC_J,  sREC_J  = mean_std(REC_J_list)
mF1_J,   sF1_J   = mean_std(F1_J_list)
mSPEC_J, sSPEC_J = mean_std(SPEC_J_list)

print("\n=== Mean ± SD across folds (TEST) — Youden-J (on TEST) ===")
print(f"{'AUC@J':>12}: {mAUC_J:.4f} ± {sAUC_J:.4f}")
print(f"{'ACC@J':>12}: {mACC_J:.4f} ± {sACC_J:.4f}")
print(f"{'Precision@J':>12}: {mPREC_J:.4f} ± {sPREC_J:.4f}")
print(f"{'Recall@J':>12}: {mREC_J:.4f} ± {sREC_J:.4f}")
print(f"{'Specificity@J':>12}: {mSPEC_J:.4f} ± {sSPEC_J:.4f}")
print(f"{'F1@J':>12}: {mF1_J:.4f} ± {sF1_J:.4f}")

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv("cv7_img_fold_metrics.csv", index=False)
print("\nSaved TEST-only per-fold metrics to cv7_img_fold_metrics.csv")


Total IDs: 42 | Pos IDs: 13 | Neg IDs: 29
TARGET_W=6 (non-Time columns)
Fixed input height after de-dup (all IDs): H=360
Normalization: NONE. De-dup: remove consecutive duplicate rows (row-wise).

=== 7-Fold CV (by patient ID) ===

--- Fold 1/7 ---
 train | ids:   32 | files:   808 | pos files:  320 | neg files:  488
   val | ids:    4 | files:    85 | pos files:    6 | neg files:   79
  test | ids:    6 | files:   131 | pos files:   45 | neg files:   86
Epoch 1/100
267/270 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - acc: 0.5378 - auc: 0.5421 - loss: 0.8756
Epoch 1: val_loss improved from inf to 0.24776, saving model to best_fold_01.h5


270/270 ━━━━━━━━━━━━━━━━━━━━ 23s 48ms/step - acc: 0.5380 - auc: 0.5425 - loss: 0.8757 - val_acc: 0.9294 - val_auc: 0.8249 - val_loss: 0.2478
Epoch 2/100
265/270 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.6322 - auc: 0.6218 - loss: 0.7960
Epoch 2: val_loss did not improve from 0.24776
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.6314 - auc: 0.6212 - loss: 0.7968 - val_acc: 0.9412 - val_auc: 0.9293 - val_loss: 0.4114
Epoch 3/100
261/270 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.5469 - auc: 0.5647 - loss: 0.8417
Epoch 3: val_loss did not improve from 0.24776
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.5471 - auc: 0.5652 - loss: 0.8416 - val_acc: 0.3529 - val_auc: 0.7911 - val_loss: 0.8502
Epoch 4/100
252/270 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.5928 - auc: 0.5982 - loss: 0.7904
Epoch 4: val_loss did not improve from 0.24776
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.5941 - auc: 0.6006 - loss: 0.7877 - val_acc: 0.3765 - val_auc: 0.9641 - val_loss: 0.7698
Epoch 5/100
2

270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.7042 - auc: 0.7589 - loss: 0.5574 - val_acc: 0.9294 - val_auc: 0.7015 - val_loss: 0.2450
Epoch 51/100
258/270 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.7025 - auc: 0.7492 - loss: 0.5634
Epoch 51: val_loss did not improve from 0.24504
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.7029 - auc: 0.7499 - loss: 0.5628 - val_acc: 0.7176 - val_auc: 0.7890 - val_loss: 0.5368
Epoch 52/100
269/270 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.6858 - auc: 0.7239 - loss: 0.5686
Epoch 52: val_loss did not improve from 0.24504
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.6858 - auc: 0.7241 - loss: 0.5685 - val_acc: 0.9294 - val_auc: 0.4989 - val_loss: 0.3223
Epoch 53/100
254/270 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.6791 - auc: 0.7275 - loss: 0.5707
Epoch 53: val_loss did not improve from 0.24504
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.6796 - auc: 0.7291 - loss: 0.5699 - val_acc: 0.8353 - val_auc: 0.7257 - val_loss: 0.3645
Epoch 54/

270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.6997 - auc: 0.7901 - loss: 0.5078 - val_acc: 0.9294 - val_auc: 0.7015 - val_loss: 0.2381
Epoch 70/100
254/270 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.7322 - auc: 0.8047 - loss: 0.4960
Epoch 70: val_loss did not improve from 0.23806
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.7307 - auc: 0.8033 - loss: 0.4985 - val_acc: 0.9176 - val_auc: 0.6245 - val_loss: 0.3019
Epoch 71/100
266/270 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.7282 - auc: 0.7859 - loss: 0.5099
Epoch 71: val_loss did not improve from 0.23806
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.7280 - auc: 0.7856 - loss: 0.5103 - val_acc: 0.9294 - val_auc: 0.5411 - val_loss: 0.2666
Epoch 72/100
255/270 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.7010 - auc: 0.7361 - loss: 0.5593
Epoch 72: val_loss did not improve from 0.23806
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.7026 - auc: 0.7389 - loss: 0.5570 - val_acc: 0.9176 - val_auc: 0.6719 - val_loss: 0.2811
Epoch 73/

Fold 1 | TEST@0.5  ACC=0.6412 | PREC=0.0000 | REC=0.0000 | F1=0.0000 | AUC=0.7848 | SPEC=0.9767 | n=131
Fold 1 | TEST@J    ACC=0.7557 | PREC=0.6444 | REC=0.6444 | F1=0.6444 | AUC=0.7848 | SPEC=0.8140 | thr=0.2385, J=0.4584
Confusion matrix (@0.5):
 [[84  2]
 [45  0]]
Classification report (@0.5):
               precision    recall  f1-score   support

           0      0.651     0.977     0.781        86
           1      0.000     0.000     0.000        45

    accuracy                          0.641       131
   macro avg      0.326     0.488     0.391       131
weighted avg      0.427     0.641     0.513       131


--- Fold 2/7 ---
 train | ids:   32 | files:   763 | pos files:  322 | neg files:  441
   val | ids:    4 | files:   149 | pos files:    4 | neg files:  145
  test | ids:    6 | files:   112 | pos files:   45 | neg files:   67
Epoch 1/100
250/255 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - acc: 0.5994 - auc: 0.5449 - loss: 0.9386
Epoch 1: val_loss improved from inf to 0.26935, s

...

277/285 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - acc: 0.5198 - auc: 0.4993 - loss: 0.9505
Epoch 1: val_loss improved from inf to 0.41305, saving model to best_fold_07.h5


285/285 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - acc: 0.5205 - auc: 0.5010 - loss: 0.9492 - val_acc: 0.8378 - val_auc: 0.8656 - val_loss: 0.4131
Epoch 2/100
272/285 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.5756 - auc: 0.5986 - loss: 0.8757
Epoch 2: val_loss did not improve from 0.41305
285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.5750 - auc: 0.5980 - loss: 0.8764 - val_acc: 0.7838 - val_auc: 0.8306 - val_loss: 0.4367
Epoch 3/100
274/285 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.5572 - auc: 0.5714 - loss: 0.8455
Epoch 3: val_loss improved from 0.41305 to 0.39657, saving model to best_fold_07.h5


285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.5574 - auc: 0.5720 - loss: 0.8454 - val_acc: 0.7568 - val_auc: 0.8548 - val_loss: 0.3966
Epoch 4/100
268/285 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.5759 - auc: 0.5776 - loss: 0.8206
Epoch 4: val_loss did not improve from 0.39657
285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.5757 - auc: 0.5780 - loss: 0.8211 - val_acc: 0.7568 - val_auc: 0.8226 - val_loss: 0.4120
Epoch 5/100
272/285 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.5473 - auc: 0.5909 - loss: 0.8245
Epoch 5: val_loss did not improve from 0.39657
285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.5492 - auc: 0.5926 - loss: 0.8223 - val_acc: 0.7568 - val_auc: 0.8145 - val_loss: 0.4081
Epoch 6/100
279/285 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc: 0.6027 - auc: 0.6429 - loss: 0.7398
Epoch 6: val_loss did not improve from 0.39657
285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.6025 - auc: 0.6426 - loss: 0.7404 - val_acc: 0.7838 - val_auc: 0.7554 - val_loss: 0.4168
Epoch 7/100
274

Fold 7 | TEST@0.5  ACC=0.9323 | PREC=0.0000 | REC=0.0000 | F1=0.0000 | AUC=0.9023 | SPEC=1.0000 | n=133
Fold 7 | TEST@J    ACC=0.8797 | PREC=0.3600 | REC=1.0000 | F1=0.5294 | AUC=0.9023 | SPEC=0.8710 | thr=0.2211, J=0.8710
Confusion matrix (@0.5):
 [[124   0]
 [  9   0]]
Classification report (@0.5):
               precision    recall  f1-score   support

           0      0.932     1.000     0.965       124
           1      0.000     0.000     0.000         9

    accuracy                          0.932       133
   macro avg      0.466     0.500     0.482       133
weighted avg      0.869     0.932     0.900       133


=== TEST metrics per fold — threshold 0.5 ===
AUC : [np.float64(0.7848), np.float64(0.3954), np.float64(0.7351), np.float64(0.7918), np.float64(0.7783), np.float64(0.8064), np.float64(0.9023)]
ACC : [0.6412, 0.5893, 0.5, 0.6608, 0.7063, 0.3822, 0.9323]
PREC: [0.0, 0.0, 0.0, 0.8, 0.0, 0.0, 0.0]
REC : [0.0, 0.0, 0.0, 0.1395, 0.0, 0.0, 0.0]
SPEC: [np.float64(0.9767), np

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
